In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
import sys
from torch.utils.data import Dataset, DataLoader, RandomSampler
import math
from collections import OrderedDict

In [2]:
#!pip install torchinfo
from torchinfo import summary

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [4]:
# Set the environment variable TOKENIZERS_PARALLELISM to 'false'
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [5]:
#!pip install transformers==4.49.0
#!pip install datasets

In [6]:
from transformers import AutoTokenizer

In [7]:
torch.__version__

'2.2.2'

In [8]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len, device):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model, device=device)
        position = torch.arange(0., max_len,
                                device=device).unsqueeze(1)
        div_term = torch.exp(torch.arange(0., d_model, 2, device=device) * -(math.log(10000.0) / d_model))
        pe_pos = torch.mul(position, div_term)
        pe[:, 0::2] = torch.sin(pe_pos)
        pe[:, 1::2] = torch.cos(pe_pos)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        out = self.pe[:, :x.size(1)].requires_grad_(False)
        return out

In [9]:
#del pe
pe_e = PositionalEncoding(d_model=768, dropout=0.1, max_len=512, device=device)
inp_tok = torch.tensor([5,8,78, 86, 78, 90, 45]).unsqueeze(0)
inp_tok, inp_tok.shape

(tensor([[ 5,  8, 78, 86, 78, 90, 45]]), torch.Size([1, 7]))

In [10]:
pe_e_e = pe_e(inp_tok)
pe_e_e, pe_e_e.shape

(tensor([[[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  ...,  1.0000e+00,
            0.0000e+00,  1.0000e+00],
          [ 8.4147e-01,  5.4030e-01,  8.2843e-01,  ...,  1.0000e+00,
            1.0243e-04,  1.0000e+00],
          [ 9.0930e-01, -4.1615e-01,  9.2799e-01,  ...,  1.0000e+00,
            2.0486e-04,  1.0000e+00],
          ...,
          [-7.5680e-01, -6.5364e-01, -6.9153e-01,  ...,  1.0000e+00,
            4.0971e-04,  1.0000e+00],
          [-9.5892e-01,  2.8366e-01, -9.8573e-01,  ...,  1.0000e+00,
            5.1214e-04,  1.0000e+00],
          [-2.7942e-01,  9.6017e-01, -4.1267e-01,  ...,  1.0000e+00,
            6.1457e-04,  1.0000e+00]]], device='cuda:0'),
 torch.Size([1, 7, 768]))

In [11]:
class Embed(nn.Module):
    def __init__(self, vocab_size, embed_dim, ctx_len, do, device):
        super().__init__()
        self.tok_layer = nn.Embedding(vocab_size, embed_dim)
        self.pos_layer = PositionalEncoding(embed_dim, do, ctx_len, device)
        self.norm_do = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(do)
            )
        self.device = device
        self.ctx_len = ctx_len

    def forward(self, inp):
        inp = inp.to(self.device)
        len_inp = inp.shape[-1]
        try:
            assert len_inp <= self.ctx_len
        except:
            print("Err..Errr. Error...Bro..Length of supplied text exceeds context length defined.Exiting the program now")
            sys.exit(1)
        tok_embed = self.tok_layer(inp)
        pos_embed = self.pos_layer(inp)
        embed_tok_pos = tok_embed + pos_embed
        embed_out = self.norm_do(embed_tok_pos)
        return embed_out

In [12]:
inp_tok = torch.tensor([5,8,78, 86, 78, 90, 45]).unsqueeze(0)
inp_tok, inp_tok.shape

(tensor([[ 5,  8, 78, 86, 78, 90, 45]]), torch.Size([1, 7]))

In [13]:
emb = Embed(100, 768, 512, 0.1, device).to(device)

In [14]:
emb_out = emb(inp_tok.to(device))
emb_out.shape, emb_out

(torch.Size([1, 7, 768]),
 tensor([[[ 0.0000,  0.7949,  1.5027,  ..., -0.0584, -0.8850,  0.4352],
          [ 0.0000,  1.2794,  0.4684,  ...,  0.0000,  0.0515,  0.3794],
          [ 0.3770, -1.3575,  0.5717,  ...,  1.0484, -0.8036,  0.2363],
          ...,
          [-1.2376, -1.4773, -1.0045,  ...,  1.0662, -0.7094,  0.2875],
          [-1.5998, -0.6224, -1.2261,  ...,  0.8126, -1.5515,  1.4202],
          [-1.8249, -0.2403, -1.6069,  ...,  2.2977,  0.4963,  0.2437]]],
        device='cuda:0', grad_fn=<NativeDropoutBackward0>))

In [15]:
summary(emb)

Layer (type:depth-idx)                   Param #
Embed                                    --
├─Embedding: 1-1                         76,800
├─PositionalEncoding: 1-2                --
│    └─Dropout: 2-1                      --
├─Sequential: 1-3                        --
│    └─LayerNorm: 2-2                    1,536
│    └─Dropout: 2-3                      --
Total params: 78,336
Trainable params: 78,336
Non-trainable params: 0

In [16]:
def att_mask(attention_mask, lookahead, cross_att, x=None):
    if cross_att:
        batch_dim = x[0]
        repeat = x[1]
    else:
        batch_dim = attention_mask.shape[0]
        repeat = len(attention_mask[0])

    mask =[]
    for i in range(batch_dim):
        am_interim = [attention_mask[i].tolist()] * repeat
        am_interim = torch.tensor(am_interim).unsqueeze(0)
        mask.append(am_interim)
    mask = torch.vstack(mask)
    if lookahead:
        inp_save = mask
        mask = torch.tril(torch.ones(mask.shape))
    mask = torch.where(mask == 0, -torch.inf, 0.0)
    return mask

In [17]:
attention_mask = torch.tensor([[1,1,1,1,0,0], [1,1,1,0,0,0]])
attention_mask.shape

torch.Size([2, 6])

In [18]:
mask  = att_mask(attention_mask, lookahead=False, cross_att=False, x=[2,9,6])

In [19]:
mask, mask.shape

(tensor([[[0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf]],
 
         [[0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf]]]),
 torch.Size([2, 6, 6]))

In [20]:
class Attention(nn.Module):
    def __init__(self, embed_dim, k_dim, do, device):
        super().__init__()
        self.embed_dim = embed_dim
        self.k_dim = k_dim
        self.query = nn.Linear(embed_dim, k_dim)
        self.key = nn.Linear(embed_dim, k_dim)
        self.value = nn.Linear(embed_dim, k_dim)
        self.att_do = nn.Dropout(do)
        self.device = device

    def forward(self, qry, ky, vlu, mask):
        q = self.query(qry)
        k = self.key(ky)
        v = self.value(vlu)
        qk = (q@k.transpose(1, 2))/(self.k_dim**0.5)
        mask = mask.to(self.device)
        qk_m = qk + mask
        qk_m_smax = torch.softmax(qk_m, dim=-1)
        qk_m_smax_do = self.att_do(qk_m_smax)
        qkv = qk_m_smax_do@v
        return qkv

In [21]:
class Attention_Block(nn.Module):
    def __init__(self, num_heads, embed_dim, k_dim, do, device):
        super().__init__()
        self.num_heads = num_heads
        self.heads_list = [Attention(embed_dim, k_dim, do, device) for i in range(num_heads)]
        self.heads = nn.ModuleList(self.heads_list)
        self.lin_do = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.Dropout(do)
        )

    def forward(self, q, k, v, mask):
        heads_list_out = [head(q, k, v, mask) for head in self.heads]
        att_head = torch.cat(heads_list_out, dim=-1)
        att_head_out = self.lin_do(att_head)
        return att_head_out

In [22]:
class Encoder_Block(nn.Module):
    def __init__(self, num_heads, embed_dim, k_dim, do, device):
        super().__init__()
        self.MHA = Attention_Block(num_heads, embed_dim, k_dim, do, device)
        self.mha_blk_end_layernorm = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim*4),
            nn.GELU(),
            nn.Linear(embed_dim*4, embed_dim),
            nn.Dropout(do)
            )
        self.enc_blk_end_layernorm = nn.LayerNorm(embed_dim)

    def forward(self, args_list):  #q, k, v, mask):
        x = args_list[0]
        mask = args_list[1]

        inp_start_att_block = x
        x = self.MHA(x, x, x, mask)

        x = x + inp_start_att_block
        x = self.mha_blk_end_layernorm(x)

        inp_start_ff_block = x
        x = self.ff(x)

        x = x + inp_start_ff_block
        x = self.enc_blk_end_layernorm(x)
        return [x, mask]

In [23]:
class Encoder(nn.Module):
    def __init__(self, num_layers, num_heads, vocab_size, embed_dim, k_dim, ctx_len, do, device):
        super().__init__()
        self.emb = Embed(vocab_size, embed_dim, ctx_len, do, device)
        self.layer_list = [Encoder_Block(num_heads, embed_dim, k_dim, do, device) for i in range(num_layers)]
        self.layers = nn.Sequential(*self.layer_list)

    def forward(self, input_ids, attention_mask):
        x = self.emb(input_ids)
        mask = att_mask(attention_mask, lookahead=False, cross_att=False, x=None)
        x = self.layers([x, mask])
        return x[0]

In [24]:
enc = Encoder(3, 6, 100, 768, 128, 100, 0.1, device)
enc.to(device)

Encoder(
  (emb): Embed(
    (tok_layer): Embedding(100, 768)
    (pos_layer): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (norm_do): Sequential(
      (0): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (1): Dropout(p=0.1, inplace=False)
    )
  )
  (layers): Sequential(
    (0): Encoder_Block(
      (MHA): Attention_Block(
        (heads): ModuleList(
          (0-5): 6 x Attention(
            (query): Linear(in_features=768, out_features=128, bias=True)
            (key): Linear(in_features=768, out_features=128, bias=True)
            (value): Linear(in_features=768, out_features=128, bias=True)
            (att_do): Dropout(p=0.1, inplace=False)
          )
        )
        (lin_do): Sequential(
          (0): Linear(in_features=768, out_features=768, bias=True)
          (1): Dropout(p=0.1, inplace=False)
        )
      )
      (mha_blk_end_layernorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (ff): Sequential(
  

In [25]:
summary(enc)

Layer (type:depth-idx)                        Param #
Encoder                                       --
├─Embed: 1-1                                  --
│    └─Embedding: 2-1                         76,800
│    └─PositionalEncoding: 2-2                --
│    │    └─Dropout: 3-1                      --
│    └─Sequential: 2-3                        --
│    │    └─LayerNorm: 3-2                    1,536
│    │    └─Dropout: 3-3                      --
├─Sequential: 1-2                             --
│    └─Encoder_Block: 2-4                     --
│    │    └─Attention_Block: 3-4              2,362,368
│    │    └─LayerNorm: 3-5                    1,536
│    │    └─Sequential: 3-6                   4,722,432
│    │    └─LayerNorm: 3-7                    1,536
│    └─Encoder_Block: 2-5                     --
│    │    └─Attention_Block: 3-8              2,362,368
│    │    └─LayerNorm: 3-9                    1,536
│    │    └─Sequential: 3-10                  4,722,432
│    │    └─LayerNor

In [26]:
inp = torch.randint(1,100, (4,6))
am = torch.ones(inp.shape)
inp.shape, am.shape

(torch.Size([4, 6]), torch.Size([4, 6]))

In [27]:
out = enc(inp.to(device), am.to(device))
out.shape

torch.Size([4, 6, 768])

In [28]:
class MLM(nn.Module):
    def __init__(self, enc, embed_dim, vocab_size):
        super().__init__()

        self.encoder = enc
        self.mlm_layer = nn.Linear(embed_dim, vocab_size)

    def forward(self, input_ids, attention_mask):
        x = self.encoder(input_ids, attention_mask)
        x = self.mlm_layer(x)
        return x

In [29]:
model_mlm = MLM(enc, 768, 36000)
model_mlm.to(device)

MLM(
  (encoder): Encoder(
    (emb): Embed(
      (tok_layer): Embedding(100, 768)
      (pos_layer): PositionalEncoding(
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm_do): Sequential(
        (0): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (1): Dropout(p=0.1, inplace=False)
      )
    )
    (layers): Sequential(
      (0): Encoder_Block(
        (MHA): Attention_Block(
          (heads): ModuleList(
            (0-5): 6 x Attention(
              (query): Linear(in_features=768, out_features=128, bias=True)
              (key): Linear(in_features=768, out_features=128, bias=True)
              (value): Linear(in_features=768, out_features=128, bias=True)
              (att_do): Dropout(p=0.1, inplace=False)
            )
          )
          (lin_do): Sequential(
            (0): Linear(in_features=768, out_features=768, bias=True)
            (1): Dropout(p=0.1, inplace=False)
          )
        )
        (mha_blk_end_layernorm): LayerNor

In [30]:
summary(model_mlm)

Layer (type:depth-idx)                             Param #
MLM                                                --
├─Encoder: 1-1                                     --
│    └─Embed: 2-1                                  --
│    │    └─Embedding: 3-1                         76,800
│    │    └─PositionalEncoding: 3-2                --
│    │    └─Sequential: 3-3                        1,536
│    └─Sequential: 2-2                             --
│    │    └─Encoder_Block: 3-4                     7,087,872
│    │    └─Encoder_Block: 3-5                     7,087,872
│    │    └─Encoder_Block: 3-6                     7,087,872
├─Linear: 1-2                                      27,684,000
Total params: 49,025,952
Trainable params: 49,025,952
Non-trainable params: 0

In [31]:
out = model_mlm(inp.to(device), am.to(device))
out, out.shape

(tensor([[[ 7.8134e-01,  4.9685e-01, -7.8744e-02,  ..., -5.6219e-01,
           -3.0834e-01, -6.7807e-01],
          [ 4.7957e-01,  9.2937e-01,  1.2288e+00,  ...,  1.5915e-01,
           -6.9048e-01, -5.6131e-01],
          [-1.6404e-01, -2.9008e-01, -6.7315e-01,  ...,  2.6473e-01,
            3.6857e-01, -7.5347e-01],
          [-1.3088e-01,  3.1274e-03, -1.3570e-01,  ..., -7.2280e-01,
            9.4652e-02, -3.7697e-01],
          [-2.9376e-01,  7.6625e-01, -4.1781e-01,  ...,  6.0205e-02,
           -5.2850e-04,  1.6111e-01],
          [ 6.9685e-01,  1.7179e-01, -5.2903e-01,  ..., -2.9139e-01,
            5.1216e-01, -7.3458e-01]],
 
         [[ 6.1739e-01, -6.8700e-02, -6.9755e-01,  ...,  2.9555e-01,
            2.1174e-01,  3.1995e-01],
          [ 8.5681e-02,  9.2743e-01, -5.5619e-01,  ...,  5.4439e-01,
           -3.6802e-01,  2.5744e-01],
          [ 4.0352e-01,  1.1474e-01, -5.9673e-01,  ...,  5.8997e-01,
            5.9198e-01, -2.3976e-01],
          [-1.6402e-01, -2.8835e-0

In [32]:
from datasets import load_dataset

In [33]:
ds_name = 'fancyzhx/ag_news'
ds = load_dataset(ds_name)

In [34]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [35]:
df_train = ds['train'].to_pandas()

In [36]:
df_test = ds['test'].to_pandas()

In [37]:
df_train.to_csv('./agnews_train', index=False)
df_test.to_csv('./agnews_test', index=False)

In [38]:
%pwd

'/home/ec2-user/SageMaker/EncoderTasks'

In [39]:
!ls -ltrh ./agnews*

-rw-rw-r-- 1 ec2-user ec2-user  28M Mar 22 10:51 ./agnews_train
-rw-rw-r-- 1 ec2-user ec2-user 1.8M Mar 22 10:51 ./agnews_test


In [40]:
tok_ckpt = 'bert-base-uncased'
orig_tokenizer = AutoTokenizer.from_pretrained(tok_ckpt)

In [41]:
orig_tokenizer

BertTokenizerFast(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [42]:
df_train.columns, df_test.columns

(Index(['text', 'label'], dtype='object'),
 Index(['text', 'label'], dtype='object'))

In [43]:
df = pd.concat([df_train, df_test], axis=0)
df

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2
...,...,...
7595,Around the world Ukrainian presidential candid...,0
7596,Void is filled with Clement With the supply of...,1
7597,Martinez leaves bitter Like Roger Clemens did ...,1
7598,5 of arthritis patients in Singapore take Bext...,2


In [44]:
text_list = df['text'].tolist()

In [45]:
len(text_list)

127600

In [46]:
text_list[:2]

["Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\\which has a reputation for making well-timed and occasionally\\controversial plays in the defense industry, has quietly placed\\its bets on another part of the market.']

In [47]:
text_list[3]

'Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\\flows from the main pipeline in southern Iraq after\\intelligence showed a rebel militia could strike\\infrastructure, an oil official said on Saturday.'

In [48]:
unq_words = sorted(list(set([w.lower() for t in text_list for w in t.split()])))

In [49]:
len(unq_words)

163946

In [50]:
text_gen = (text_list[i:i+1000] for i in range(0, len(text_list), 1000))

In [51]:
text_gen

<generator object <genexpr> at 0x7feb78258c10>

In [52]:
news_tokenizer = orig_tokenizer.train_new_from_iterator(text_gen, 9000) #vocab size of 9000

In [53]:
len(orig_tokenizer.tokenize(text_list[0]))

39

In [54]:
len(news_tokenizer.tokenize(text_list[0]))

42

In [55]:
news_tokenizer

BertTokenizerFast(name_or_path='bert-base-uncased', vocab_size=9000, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [56]:
#we placed a limit on vocab size to make model size small so that we can train easy peasy

In [57]:
df['text_len'] = df['text'].apply(lambda x: len(x))

In [58]:
df.describe()

,label,text_len
count,127600.000000,127600.000000
mean,1.500000,236.407343
std,1.118038,66.438756
min,0.000000,100.000000
25%,0.750000,196.000000
50%,1.500000,232.000000
75%,2.250000,266.000000
max,3.000000,1012.000000


In [59]:
class NEWS_DS(Dataset):
    def __init__(self, df):
        self.data = df['text']

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        data = self.data.iloc[idx]
        return data

In [60]:
news_ds = NEWS_DS(df)

In [61]:
len(news_ds)

127600

In [62]:
news_ds[2]

"Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\\about the economy and the outlook for earnings are expected to\\hang over the stock market next week during the depth of the\\summer doldrums."

In [63]:
news_tokenizer.encode(news_tokenizer.mask_token)

[2, 4, 3]

In [64]:
orig_tokenizer.encode(news_tokenizer.mask_token)

[101, 103, 102]

In [65]:
def bernoulli_true_false(p):
    # Create a Bernoulli distribution with probability p
    bernoulli_dist = torch.distributions.Bernoulli(torch.tensor([p]))
    # Sample from this distribution and convert 1 to True and 0 to False
    return bernoulli_dist.sample().item() == 1

In [66]:
torch.randint(0, 9000, size=(1,)).item() #.squeeze().item()

3843

In [67]:
def Masking(token, vocab_size):
    # Decide whether to mask this token (15% chance)
    mask = bernoulli_true_false(0.15)

    # If mask is False, immediately return with '[PAD]' label
    if not mask:
        status = "nomask"
        return token, '[PAD]', status

    # If mask is True, proceed with further operations
    # Randomly decide on an operation (10% chance each)
    replace_random = bernoulli_true_false(0.1)
    leave_as_is = bernoulli_true_false(0.1)
    if replace_random:
        _token = torch.randint(0, vocab_size, size=(1,)).item()
        label = token
        status = "random"
    elif leave_as_is:
        _token = token
        label = token
        status = "asis"
    else:
        _token = '[MASK]'
        label = token
        status = "masked"
        
    return _token, label, status

In [68]:
vocab_size = news_tokenizer.vocab_size
vocab_size

9000

In [69]:
text_list[3]

'Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\\flows from the main pipeline in southern Iraq after\\intelligence showed a rebel militia could strike\\infrastructure, an oil official said on Saturday.'

In [70]:
news_tokenizer(text_list[3], max_length=256, truncation=True)

{'input_ids': [2, 366, 4363, 70, 426, 4198, 241, 1459, 1823, 5900, 11, 231, 12, 231, 15, 2474, 310, 4363, 104, 426, 5669, 32, 7206, 70, 241, 102, 1459, 5900, 108, 1823, 366, 264, 32, 3596, 2189, 34, 2280, 5430, 592, 1746, 32, 6156, 14, 117, 426, 560, 221, 131, 696, 16, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [71]:
inp = news_tokenizer(text_list[:8]).input_ids
inp

[[2,
  1405,
  134,
  16,
  3934,
  267,
  1104,
  523,
  469,
  102,
  1741,
  11,
  231,
  12,
  231,
  15,
  1551,
  15,
  1066,
  148,
  14,
  1405,
  1456,
  10,
  52,
  6859,
  1187,
  900,
  32,
  4083,
  115,
  5665,
  161,
  15,
  1997,
  6516,
  70,
  14,
  318,
  7661,
  1900,
  381,
  16,
  3],
 [2,
  3668,
  3199,
  2858,
  2057,
  2990,
  8081,
  11,
  231,
  12,
  231,
  15,
  2587,
  2364,
  1286,
  3668,
  3199,
  563,
  14,
  32,
  637,
  230,
  34,
  8389,
  135,
  1739,
  1570,
  15,
  552,
  74,
  132,
  3492,
  5278,
  702,
  32,
  3692,
  4694,
  108,
  102,
  1785,
  1076,
  14,
  230,
  7555,
  4970,
  32,
  209,
  1291,
  70,
  131,
  1009,
  581,
  115,
  102,
  667,
  16,
  3],
 [2,
  426,
  132,
  1414,
  7159,
  605,
  10,
  2279,
  11,
  231,
  12,
  231,
  15,
  5225,
  1647,
  607,
  4322,
  3287,
  32,
  488,
  102,
  1414,
  132,
  102,
  2279,
  135,
  1197,
  318,
  947,
  111,
  32,
  5097,
  275,
  102,
  1089,
  667,
  598,
  458,
  1050,
  102,


In [72]:
inp_mask  = [Masking(i, vocab_size) for il in inp for i in il]
len(inp_mask)

429

In [73]:
no_mask = 0
masked = 0
random = 0
asis = 0
for i, t, s in inp_mask:
    if s == "nomask":
        no_mask+=1
    elif s == 'masked':
        masked+=1
    elif s == "random":
        random+=1
    else:
        asis+=1
        

In [74]:
no_mask, masked, random, asis

(366, 53, 3, 7)

In [75]:
429*0.15

64.35

In [76]:
429*0.15*0.8

51.48

In [77]:
429*0.15*0.1

6.435

In [78]:
#so masking is woring good

In [79]:
class NEWS_DS(Dataset):
    def __init__(self, df):
        self.data = df['text']

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data.iloc[idx]
        return text

In [80]:
news_tokenizer.vocab['[MASK]'], news_tokenizer.vocab['[PAD]']

(4, 0)

In [81]:
def Masking(token, vocab_size):
    # Decide whether to mask this token (15% chance)
    mask = bernoulli_true_false(0.15)

    # If mask is False, immediately return with '[PAD]' label
    if not mask:
        return token, -100

    # If mask is True, proceed with further operations
    # Randomly decide on an operation (10% chance each)
    replace_random = bernoulli_true_false(0.1)
    leave_as_is = bernoulli_true_false(0.1)
    label = token
    if replace_random:
        _token = torch.randint(0, vocab_size, size=(1,)).item()
    elif leave_as_is:
        _token = token
    else:
        _token = 4

    return _token, label

In [82]:
news_tokenizer.vocab[news_tokenizer.pad_token], news_tokenizer.vocab[news_tokenizer.mask_token]

(0, 4)

In [83]:
pad_tokenid = [news_tokenizer.vocab[news_tokenizer.pad_token]]
type(pad_tokenid[0])

int

In [84]:
def collate_fn(batch, tokenizer):
    vocab_size = tokenizer.vocab_size
    pad_tokenid = [tokenizer.vocab[news_tokenizer.pad_token]]
    ignore_loss_id = [-100]  #ignore loss
    ignore_token_id = [0]   #attention mask filler
    x_input_ids, x_am = [], []
    lab_list = []

    for x in batch:
        tokenized = tokenizer(x, max_length=256, truncation=True)
        inp_ids = []
        label_list = []
        for x in tokenized.input_ids:
            tok, lbl = Masking(x, vocab_size)
            inp_ids.append(tok)
            label_list.append(lbl)
        am = tokenized.attention_mask
        x_input_ids.append(inp_ids)
        x_am.append(am)
        lab_list.append(label_list)

    len_enc = [len(l) for l in x_input_ids]
    max_len = max(len_enc)

    def pad_tokens_stack(in_list, max_len, padding):
        list_tensor = []
        for item in in_list:
            len_item = len(item)
            deficit = max_len - len_item
            if deficit > 0:
                item = item + padding*deficit
            list_tensor.append(torch.tensor(item))
        item_ids = torch.vstack(list_tensor)
        return item_ids

    input_ids = pad_tokens_stack(x_input_ids, max_len, pad_tokenid)
    attention_mask = pad_tokens_stack(x_am, max_len, ignore_token_id)
    labels =  pad_tokens_stack(lab_list, max_len, ignore_loss_id)

    return {'input_ids': input_ids, 'attention_mask': attention_mask}, labels

In [85]:
train_ds = NEWS_DS(df_train)
test_ds = NEWS_DS(df_test)

In [86]:
len(train_ds), len(test_ds)

(120000, 7600)

In [87]:
torch.cuda.empty_cache()

In [88]:
from functools import partial
import os
wrapper_collate_fn = partial(
    collate_fn,
    tokenizer=news_tokenizer,
    )a

train_dl = DataLoader(train_ds, shuffle=False, batch_size=128,
                      num_workers=os.cpu_count(), collate_fn=wrapper_collate_fn)
test_dl = DataLoader(test_ds, shuffle=False, batch_size=128,
                      num_workers=os.cpu_count(), collate_fn=wrapper_collate_fn)

In [89]:
train_iter = iter(train_dl)

In [90]:
i, l = next(train_iter)

In [91]:
unravel = [t for i in l.tolist() for t in i]

In [92]:
len(unravel)

16256

In [93]:
valid_items = [i for i in unravel if i > -100]
valid_items

[16,
 267,
 1066,
 1405,
 32,
 1997,
 7661,
 1900,
 381,
 2858,
 3668,
 563,
 8389,
 16,
 2,
 132,
 231,
 5225,
 1647,
 16,
 2,
 231,
 32,
 70,
 16,
 607,
 111,
 15,
 1375,
 5702,
 1269,
 1509,
 1874,
 2344,
 742,
 271,
 2409,
 3,
 1313,
 217,
 1330,
 712,
 160,
 727,
 108,
 186,
 135,
 221,
 6672,
 160,
 34,
 16,
 37,
 16,
 51,
 102,
 1356,
 6529,
 1286,
 14,
 34,
 6016,
 265,
 2310,
 357,
 3346,
 2310,
 102,
 2165,
 241,
 16,
 3,
 11,
 14,
 4083,
 6516,
 70,
 1900,
 426,
 275,
 193,
 135,
 3474,
 1233,
 12,
 2246,
 16,
 660,
 8229,
 221,
 696,
 607,
 3266,
 3776,
 69,
 4265,
 14,
 1384,
 769,
 102,
 981,
 283,
 2186,
 240,
 462,
 264,
 102,
 5942,
 5197,
 496,
 11,
 102,
 6862,
 131,
 34,
 54,
 16,
 108,
 6678,
 5534,
 111,
 1014,
 2624,
 1312,
 7996,
 325,
 1587,
 168,
 15,
 15,
 34,
 581,
 115,
 523,
 410,
 1038,
 667,
 1263,
 357,
 4583,
 3131,
 484,
 10,
 504,
 1642,
 175,
 3,
 102,
 219,
 646,
 8,
 3081,
 162,
 1843,
 14,
 1510,
 34,
 892,
 4045,
 102,
 723,
 10,
 209,
 3292,
 1

In [94]:
len(valid_items)

1116

In [95]:
16256*.15

2438.4

In [96]:
# so this is all good. next we need to train hte model

In [97]:
from torch.optim import AdamW

In [98]:
def train_model(num_epochs, lrate, model, dl, device, vocab_size):
    epochs = num_epochs
    lr = lrate
    fn_loss = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=lr)
    grad_accum_steps = 1 #args.grad_accum_steps # 4 - previous static value
    model = model.to(device)

    for i in range(epochs):
        loss_epoch = 0
        n_step = 0
        grad_accum_counter = 1

        for inputs in dl:
            data = inputs[0]
            label = inputs[1]
            batch_size = label.shape[0]
            ctx_size = label.shape[1]
            data = {i: k.to(device) for i, k in data.items()}
            label = label.to(device)
            out = model(**data)
            out = out.view(batch_size * ctx_size, vocab_size)
            label = label.view(batch_size * ctx_size)
            loss = fn_loss(out, label)
            loss = loss / grad_accum_steps
            loss.backward()
            if grad_accum_counter == grad_accum_steps:
                #logger.info(f'Steps: {grad_accum_counter}, adjusting learnable params now')
                optimizer.step()
                optimizer.zero_grad()
                grad_accum_counter = 0
            loss_epoch = loss_epoch + (loss.item() * batch_size * grad_accum_steps)
            if n_step % 100 == 0:
                print(f'Step: {n_step}, Loss: {loss.item()}')
            grad_accum_counter += 1
            n_step += 1
            ##update so that if remaining dataloader runs are less than grad accum steps then at the last run i should
            #gradient update
            #print(f'Completed minibatch loop')
        average_loss = loss_epoch/len(dl.dataset)
        perplexity = math.exp(average_loss) 
        print(f'Epoch: {i} -- Average loss: {average_loss}')
        print(f'Epoch: {i} -- Perplexity: {perplexity}')

    return model, optimizer, average_loss, perplexity


In [99]:
vocab_size = news_tokenizer.vocab_size
vocab_size

9000

In [100]:
BroEncoder = Encoder(num_layers=6, num_heads=8, vocab_size=vocab_size, 
                     embed_dim=256, k_dim=32, ctx_len=256, do=0.1, device=device)

In [101]:
BroEncoder

Encoder(
  (emb): Embed(
    (tok_layer): Embedding(9000, 256)
    (pos_layer): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (norm_do): Sequential(
      (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (1): Dropout(p=0.1, inplace=False)
    )
  )
  (layers): Sequential(
    (0): Encoder_Block(
      (MHA): Attention_Block(
        (heads): ModuleList(
          (0-7): 8 x Attention(
            (query): Linear(in_features=256, out_features=32, bias=True)
            (key): Linear(in_features=256, out_features=32, bias=True)
            (value): Linear(in_features=256, out_features=32, bias=True)
            (att_do): Dropout(p=0.1, inplace=False)
          )
        )
        (lin_do): Sequential(
          (0): Linear(in_features=256, out_features=256, bias=True)
          (1): Dropout(p=0.1, inplace=False)
        )
      )
      (mha_blk_end_layernorm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (ff): Sequential(
    

In [102]:
summary(BroEncoder)

Layer (type:depth-idx)                        Param #
Encoder                                       --
├─Embed: 1-1                                  --
│    └─Embedding: 2-1                         2,304,000
│    └─PositionalEncoding: 2-2                --
│    │    └─Dropout: 3-1                      --
│    └─Sequential: 2-3                        --
│    │    └─LayerNorm: 3-2                    512
│    │    └─Dropout: 3-3                      --
├─Sequential: 1-2                             --
│    └─Encoder_Block: 2-4                     --
│    │    └─Attention_Block: 3-4              263,168
│    │    └─LayerNorm: 3-5                    512
│    │    └─Sequential: 3-6                   525,568
│    │    └─LayerNorm: 3-7                    512
│    └─Encoder_Block: 2-5                     --
│    │    └─Attention_Block: 3-8              263,168
│    │    └─LayerNorm: 3-9                    512
│    │    └─Sequential: 3-10                  525,568
│    │    └─LayerNorm: 3-11      

In [103]:
Bro_MLM = MLM(BroEncoder, 256, vocab_size)
Bro_MLM

MLM(
  (encoder): Encoder(
    (emb): Embed(
      (tok_layer): Embedding(9000, 256)
      (pos_layer): PositionalEncoding(
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm_do): Sequential(
        (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (1): Dropout(p=0.1, inplace=False)
      )
    )
    (layers): Sequential(
      (0): Encoder_Block(
        (MHA): Attention_Block(
          (heads): ModuleList(
            (0-7): 8 x Attention(
              (query): Linear(in_features=256, out_features=32, bias=True)
              (key): Linear(in_features=256, out_features=32, bias=True)
              (value): Linear(in_features=256, out_features=32, bias=True)
              (att_do): Dropout(p=0.1, inplace=False)
            )
          )
          (lin_do): Sequential(
            (0): Linear(in_features=256, out_features=256, bias=True)
            (1): Dropout(p=0.1, inplace=False)
          )
        )
        (mha_blk_end_layernorm): LayerNorm(

In [104]:
summary(Bro_MLM)

Layer (type:depth-idx)                             Param #
MLM                                                --
├─Encoder: 1-1                                     --
│    └─Embed: 2-1                                  --
│    │    └─Embedding: 3-1                         2,304,000
│    │    └─PositionalEncoding: 3-2                --
│    │    └─Sequential: 3-3                        512
│    └─Sequential: 2-2                             --
│    │    └─Encoder_Block: 3-4                     789,760
│    │    └─Encoder_Block: 3-5                     789,760
│    │    └─Encoder_Block: 3-6                     789,760
│    │    └─Encoder_Block: 3-7                     789,760
│    │    └─Encoder_Block: 3-8                     789,760
│    │    └─Encoder_Block: 3-9                     789,760
├─Linear: 1-2                                      2,313,000
Total params: 9,356,072
Trainable params: 9,356,072
Non-trainable params: 0

In [ ]:
%%time
model, optimizer, average_loss, perplexity = train_model(10, 0.0004, Bro_MLM, train_dl, device, vocab_size)

Step: 0, Loss: 9.216163635253906


In [106]:
model

MLM(
  (encoder): Encoder(
    (emb): Embed(
      (tok_layer): Embedding(9000, 256)
      (pos_layer): PositionalEncoding(
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm_do): Sequential(
        (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (1): Dropout(p=0.1, inplace=False)
      )
    )
    (layers): Sequential(
      (0): Encoder_Block(
        (MHA): Attention_Block(
          (heads): ModuleList(
            (0-7): 8 x Attention(
              (query): Linear(in_features=256, out_features=32, bias=True)
              (key): Linear(in_features=256, out_features=32, bias=True)
              (value): Linear(in_features=256, out_features=32, bias=True)
              (att_do): Dropout(p=0.1, inplace=False)
            )
          )
          (lin_do): Sequential(
            (0): Linear(in_features=256, out_features=256, bias=True)
            (1): Dropout(p=0.1, inplace=False)
          )
        )
        (mha_blk_end_layernorm): LayerNorm(

In [107]:
torch.save({
        'epoch': 10,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': average_loss,
        'device': device
        }, './mlmckpt')


In [108]:
average_loss

3.4532971148173015

In [109]:
%pwd

'/home/ec2-user/SageMaker/EncoderTasks'

In [110]:
!ls -ltrh

total 138M
-rw-rw-r-- 1 ec2-user ec2-user  28M Mar 22 10:51 agnews_train
-rw-rw-r-- 1 ec2-user ec2-user 1.8M Mar 22 10:51 agnews_test
-rw-rw-r-- 1 ec2-user ec2-user 108M Mar 22 13:45 mlmckpt
-rw-rw-r-- 1 ec2-user ec2-user 133K Mar 22 13:48 Encoder_MLM.ipynb


In [112]:
device, perplexity

('cuda', 31.60442412716105)

In [115]:
model.eval()
fn_loss = nn.CrossEntropyLoss()
model = model.to(device)
loss_epoch = 0
for inputs in train_dl:
    data = inputs[0]
    label = inputs[1]
    batch_size = label.shape[0]
    ctx_size = label.shape[1]
    data = {i: k.to(device) for i, k in data.items()}
    label = label.to(device)
    with torch.no_grad():
        out = model(**data)
    out = out.view(batch_size * ctx_size, vocab_size)
    label = label.view(batch_size * ctx_size)
    loss = fn_loss(out, label)
    loss_epoch = loss_epoch + (loss.item() * batch_size)
average_loss = loss_epoch/len(train_dl.dataset)
perplexity = math.exp(average_loss) 
print(f'Epoch: {i} -- Average loss: {average_loss}')
print(f'Epoch: {i} -- Perplexity: {perplexity}')

Epoch: {'input_ids': tensor([[   2, 1405,  134,  ...,    0,    0,    0],
        [   2, 3668, 3199,  ...,    0,    0,    0],
        [   4,  426,    4,  ...,    0,    0,    0],
        ...,
        [   2, 1759, 4627,  ...,    0,    0,    0],
        [   4,  102,  598,  ...,    0,    0,    0],
        [   4, 1136,   14,  ...,    0,    0,    0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])} -- Average loss: 3.151042181523641
Epoch: {'input_ids': tensor([[   2, 1405,  134,  ...,    0,    0,    0],
        [   2, 3668, 3199,  ...,    0,    0,    0],
        [   4,  426,    4,  ...,    0,    0,    0],
        ...,
        [   2, 1759, 4627,  ...,    0,    0,    0],
        [   4,  102,  598,  ...,    0,    0,    0],
        [   4, 1136,   14,  ...,    0,    0,    0]]), 'attention_mask': tensor([[1, 1,

In [116]:
model.eval()
fn_loss = nn.CrossEntropyLoss()
model = model.to(device)
loss_epoch = 0
for inputs in test_dl:
    data = inputs[0]
    label = inputs[1]
    batch_size = label.shape[0]
    ctx_size = label.shape[1]
    data = {i: k.to(device) for i, k in data.items()}
    label = label.to(device)
    with torch.no_grad():
        out = model(**data)
    out = out.view(batch_size * ctx_size, vocab_size)
    label = label.view(batch_size * ctx_size)
    loss = fn_loss(out, label)
    loss_epoch = loss_epoch + (loss.item() * batch_size)
average_loss = loss_epoch/len(train_dl.dataset)
perplexity = math.exp(average_loss) 
print(f'Test Average loss: {average_loss}')
print(f'Test Perplexity: {perplexity}')

Test Average loss: 0.20677252238591512
Test Perplexity: 1.2297028101009242


In [146]:
train_ds[12]

'Non-OPEC Nations Should Up Output-Purnomo  JAKARTA (Reuters) - Non-OPEC oil exporters should consider  increasing output to cool record crude prices, OPEC President  Purnomo Yusgiantoro said on Sunday.'

In [120]:
text = "There is so much of the [MASK] pollution nowadays"
tok = news_tokenizer(text, return_tensors='pt')
input_ids = tok.input_ids.to(device)
attention_mask = tok.attention_mask.to(device)
input_ids.shape, attention_mask.shape
with torch.no_grad():
    out = model(input_ids=input_ids, attention_mask=attention_mask)

In [125]:
input_ids

tensor([[   2, 1038,  174,  492, 1420,  115,  102,    4, 7094,  902, 3273, 2051,
            3]], device='cuda:0')

In [121]:
out.shape

torch.Size([1, 13, 9000])

In [122]:
out_max = torch.argmax(out, dim=-1)
out_max

tensor([[   2, 1038,  174,  492, 1420,  115,  102,  186,  115,  902, 1427, 2051,
            3]], device='cuda:0')

In [129]:
news_tokenizer.decode([out_max[0][7].item()])

'new'

In [130]:
text = "I saw a [MASK] building today"
tok = news_tokenizer(text, return_tensors='pt')
input_ids = tok.input_ids.to(device)
attention_mask = tok.attention_mask.to(device)
input_ids.shape, attention_mask.shape
with torch.no_grad():
    out = model(input_ids=input_ids, attention_mask=attention_mask)

In [131]:
input_ids

tensor([[   2,   42, 3740,   34,    4, 2900,  625,    3]], device='cuda:0')

In [133]:
out_max = torch.argmax(out, dim=-1)
news_tokenizer.decode([out_max[0][4].item()])

'whole'

In [143]:
text = "Oil prices [MASK] to all-time record"
tok = news_tokenizer(text, return_tensors='pt')
input_ids = tok.input_ids.to(device)
mask_index = input_ids[0].tolist().index(4)
attention_mask = tok.attention_mask.to(device)
input_ids.shape, attention_mask.shape
with torch.no_grad():
    out = model(input_ids=input_ids, attention_mask=attention_mask)
out_max = torch.argmax(out, dim=-1)
news_tokenizer.decode([out_max[0][mask_index].item()])

'fall'

In [144]:
text = "Calif. Aims to [MASK] Farm-Related Smog"
tok = news_tokenizer(text, return_tensors='pt')
input_ids = tok.input_ids.to(device)
mask_index = input_ids[0].tolist().index(4)
attention_mask = tok.attention_mask.to(device)
input_ids.shape, attention_mask.shape
with torch.no_grad():
    out = model(input_ids=input_ids, attention_mask=attention_mask)
out_max = torch.argmax(out, dim=-1)
news_tokenizer.decode([out_max[0][mask_index].item()])

'buy'

In [147]:
text = "Non-OPEC oil exporters should consider  [MASK] output to cool record crude prices"
tok = news_tokenizer(text, return_tensors='pt')
input_ids = tok.input_ids.to(device)
mask_index = input_ids[0].tolist().index(4)
attention_mask = tok.attention_mask.to(device)
input_ids.shape, attention_mask.shape
with torch.no_grad():
    out = model(input_ids=input_ids, attention_mask=attention_mask)
out_max = torch.argmax(out, dim=-1)
news_tokenizer.decode([out_max[0][mask_index].item()])

'oil'

In [1]:
26000/128

203.125